# Model Definition — FL-IDS Pipeline

**Project:** Evaluating Federated Learning for Intrusion Detection in Industrial IoT  
**Module:** `src/model.py`  

---

## Purpose

This notebook walks through the neural network architecture, verifies it works
end-to-end with the preprocessed data, and documents the design decisions.

The architecture is **intentionally simple** — a feedforward neural network with
two hidden layers. The research contribution of this thesis is the *federated
learning evaluation*, not the model architecture itself.

## Architecture (from Proposal)

```
Input (17 features)
  → Linear(128) → ReLU → Dropout(0.3)
  → Linear(64)  → ReLU → Dropout(0.3)
  → Linear(8)   [raw logits — softmax is inside CrossEntropyLoss]
```

## Key Design Decisions

| Decision | Choice | Rationale |
|----------|--------|----------|
| Architecture | Feedforward (MLP) | Simple, fast, small model size for FL communication |
| Hidden sizes | 128 → 64 | Sufficient capacity for 17-feature, 8-class task |
| Activation | ReLU | Standard, no vanishing gradient issues |
| Regularization | Dropout(0.3) | Prevents overfitting on small per-client datasets |
| Loss | CrossEntropyLoss | Standard for multi-class; supports class weights |
| Optimizer | SGD | Standard for FedAvg (matches McMahan et al.) |
| No Softmax in forward() | — | PyTorch's CrossEntropyLoss applies it internally |

---
## Step 0 — Imports

In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
import sys

sys.path.insert(0, os.path.abspath('../src/'))

from model import (
    IDSNet,
    get_model_params,
    set_model_params,
    count_parameters,
    compute_model_size_bytes,
    create_model,
)
from preprocessing import load_processed_data, apply_scaler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

PyTorch version: 2.10.0+cu128
Device: cpu


---
## Step 1 — Instantiate and Inspect the Model

Let's create the model and look at its structure. Every client and the
server will use this exact same architecture — the only thing that differs
across clients is the data they train on.

In [6]:
model = create_model(num_features=17, num_classes=8, device=DEVICE)

print(f"Device: {DEVICE}")
print(f"\nModel architecture:")
print(model)

print(f"\nPer-layer parameter breakdown:")
print(f"{'Layer':<35s} {'Shape':<20s} {'Params':>8s}")
print("─" * 65)
layer_total = 0
for name, param in model.named_parameters():
    n = param.numel()
    layer_total += n
    print(f"  {name:<33s} {str(list(param.shape)):<20s} {n:>8,}")
print("─" * 65)
print(f"  {'TOTAL':<33s} {'':20s} {layer_total:>8,}")

Device: cpu

Model architecture:
IDSNet(
  (network): Sequential(
    (0): Linear(in_features=17, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=8, bias=True)
  )
)

Per-layer parameter breakdown:
Layer                               Shape                  Params
─────────────────────────────────────────────────────────────────
  network.0.weight                  [128, 17]               2,176
  network.0.bias                    [128]                     128
  network.3.weight                  [64, 128]               8,192
  network.3.bias                    [64]                       64
  network.6.weight                  [8, 64]                   512
  network.6.bias                    [8]                         8
─────────────────────────────────────────────────────────────────
  TOTAL    

---
## Step 2 — Communication Cost Analysis

Understanding where the parameters live helps interpret communication costs.
Each FedAvg round, every participating client uploads and downloads
the full parameter set — so model size directly impacts bandwidth.

In [7]:
total_params, trainable_params = count_parameters(model)
size = compute_model_size_bytes(model)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (float32): {size:,} bytes = {size / 1024:.1f} KB")

print(f"\nCommunication cost per round (all clients participate):")
print(f"{'K':>5s}  {'Upload':>12s}  {'Download':>12s}  {'Total':>12s}")
print("─" * 50)
for K in [5, 10]:
    upload   = size * K
    download = size * K
    total    = upload + download
    print(f"{K:>5d}  {upload / 1024:>10.1f} KB  {download / 1024:>10.1f} KB  {total / 1024:>10.1f} KB")

print(f"\nCommunication cost over full experiment (R rounds, K=5):")
print(f"{'R':>5s}  {'Total Cost':>12s}")
print("─" * 25)
for R in [10, 25, 50, 100]:
    total = size * 5 * 2 * R
    print(f"{R:>5d}  {total / (1024*1024):>10.2f} MB")

Total parameters:     11,080
Trainable parameters: 11,080
Model size (float32): 44,320 bytes = 43.3 KB

Communication cost per round (all clients participate):
    K        Upload      Download         Total
──────────────────────────────────────────────────
    5       216.4 KB       216.4 KB       432.8 KB
   10       432.8 KB       432.8 KB       865.6 KB

Communication cost over full experiment (R rounds, K=5):
    R    Total Cost
─────────────────────────
   10        4.23 MB
   25       10.57 MB
   50       21.13 MB
  100       42.27 MB


---
## Step 3 — Forward Pass Smoke Test

Verify the model can process a batch of data end-to-end. We'll create a
random batch with the correct shape (batch_size=8, num_features=17) and
check that the output has shape (8, 8) — one logit per class.

In [8]:
# ── Forward pass (training mode — returns raw logits) ─────────────
x = torch.randn(8, 17).to(DEVICE)

model.train()
logits = model(x)
print(f"Input shape:   {list(x.shape)}")
print(f"Output shape:  {list(logits.shape)}  (batch_size x num_classes)")
print(f"Sample logits: {logits[0].detach().cpu().numpy().round(3)}")

assert logits.shape == (8, 8), f"Unexpected output shape: {logits.shape}"

# ── Predictions (apply softmax manually for inference) ─────────────
model.eval()
with torch.no_grad():
    logits_eval = model(x)
    probs = torch.softmax(logits_eval, dim=1)
    preds = probs.argmax(dim=1)

print(f"\nPredictions:   {preds.cpu().tolist()}")
print(f"Prob sums:     {probs.sum(dim=1).cpu().numpy().round(6)}  (should all be 1.0)")
print(f"Sample probs:  {probs[0].cpu().numpy().round(4)}")
print(f"\nForward pass smoke test PASSED ✓")

Input shape:   [8, 17]
Output shape:  [8, 8]  (batch_size x num_classes)
Sample logits: [ 4.887 -0.69  -1.48   3.168 -2.308 -0.473  2.271 -0.521]

Predictions:   [0, 5, 4, 7, 5, 3, 4, 4]
Prob sums:     [1. 1. 1. 1. 1. 1. 1. 1.]  (should all be 1.0)
Sample probs:  [0.5221 0.0241 0.0124 0.3329 0.0368 0.0146 0.0338 0.0234]

Forward pass smoke test PASSED ✓


---
## Step 4 — Test with Real Preprocessed Data

Now let's load the actual preprocessed dataset and run a batch through
the model. This confirms the full pipeline works: preprocessing → model.

In [9]:
DATA_PATH = "../data/processed/datasense_preprocessed.csv"

X, y, device_names = load_processed_data(DATA_PATH)
print(f"Loaded dataset: X={X.shape}, y={y.shape}")

# ── Apply scaler (fit on full data just for this test) ─────────────
X_scaled, _, scaler = apply_scaler(X)

# ── Create a batch from real data ─────────────────────────────────
batch_indices = np.random.choice(len(X_scaled), size=64, replace=False)
x_batch = torch.tensor(X_scaled[batch_indices], dtype=torch.float32).to(DEVICE)
y_batch = torch.tensor(y[batch_indices], dtype=torch.long).to(DEVICE)

model.eval()
with torch.no_grad():
    logits = model(x_batch)
    preds = logits.argmax(dim=1)

print(f"\nBatch: {x_batch.shape[0]} real samples")
print(f"Output shape: {logits.shape}")
print(f"Predictions (first 10): {preds[:10].cpu().numpy()}")
print(f"True labels (first 10): {y_batch[:10].cpu().numpy()}")

# ── Accuracy on this random batch (untrained model, expect ~12.5%) ─
acc = (preds == y_batch).float().mean().item()
print(f"\nUntrained accuracy on batch: {acc:.2%} (random chance = 12.5%)")
print(f"Real data forward pass PASSED ✓")

Loaded dataset: X=(30030, 17), y=(30030,)

Batch: 64 real samples
Output shape: torch.Size([64, 8])
Predictions (first 10): [4 5 4 4 4 4 4 4 5 4]
True labels (first 10): [3 6 0 0 0 4 6 6 2 6]

Untrained accuracy on batch: 3.12% (random chance = 12.5%)
Real data forward pass PASSED ✓


---
## Step 5 — Test Loss Computation with Class Weights

Verify that `CrossEntropyLoss` works with our class weights. The weighted
loss penalizes mistakes on rare classes (bruteforce, web) more heavily
than mistakes on common classes (benign, recon).

In [10]:
import json
from preprocessing import compute_class_weights_from_labels

# ── Load or compute class weights ─────────────────────────────────
LABEL_CFG_PATH = "../data/processed/label_config.json"

if os.path.exists(LABEL_CFG_PATH):
    with open(LABEL_CFG_PATH, 'r') as f:
        label_cfg = json.load(f)
    weights = torch.tensor(
        [label_cfg['class_weights'][str(i)] for i in range(8)],
        dtype=torch.float32
    ).to(DEVICE)
    print("Class weights loaded from label_config.json:")
else:
    weights = torch.tensor(
        compute_class_weights_from_labels(y),
        dtype=torch.float32
    ).to(DEVICE)
    print("Class weights computed from labels:")

for i, w in enumerate(weights):
    print(f"  Class {i}: weight = {w:.4f}")

# ── Compute loss ──────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=weights)

model.train()
logits = model(x_batch)
loss = criterion(logits, y_batch)

print(f"\nWeighted CrossEntropyLoss on batch: {loss.item():.4f}")
assert not torch.isnan(loss), "Loss is NaN!"
assert not torch.isinf(loss), "Loss is infinite!"

# ── Check gradients exist ─────────────────────────────────────────
loss.backward()
has_grads = all(p.grad is not None for p in model.parameters())
print(f"All gradients computed: {has_grads}")
print(f"Loss computation + backward PASSED ✓")

Class weights loaded from label_config.json:
  Class 0: weight = 0.2744
  Class 1: weight = 10.3125
  Class 2: weight = 1.1607
  Class 3: weight = 1.1444
  Class 4: weight = 2.5995
  Class 5: weight = 2.5518
  Class 6: weight = 0.6251
  Class 7: weight = 6.8003

Weighted CrossEntropyLoss on batch: 3.6838
All gradients computed: True
Loss computation + backward PASSED ✓


---
## Step 6 — Test SGD Optimizer Step

A complete training step to confirm weights update. This is the
inner loop that each FL client runs during local training:
zero_grad → forward → loss → backward → step.

In [11]:
# ── Fresh model for clean test ────────────────────────────────────
model_train = create_model(device=DEVICE)
optimizer = torch.optim.SGD(model_train.parameters(), lr=0.01)

# Record weights before
params_before = get_model_params(model_train)

# One training step
model_train.train()
x = torch.randn(32, 17).to(DEVICE)
y_fake = torch.randint(0, 8, (32,)).to(DEVICE)

optimizer.zero_grad()
logits = model_train(x)
loss = criterion(logits, y_fake)
loss.backward()
optimizer.step()

# Record weights after
params_after = get_model_params(model_train)

# Verify weights changed
print(f"Loss: {loss.item():.4f}")
print(f"\nWeight changes after one SGD step (lr=0.01):")
for i, (before, after) in enumerate(zip(params_before, params_after)):
    diff = np.abs(after - before).mean()
    print(f"  Layer {i}: mean |delta| = {diff:.6f}")

weights_changed = any(
    not np.allclose(a, b) for a, b in zip(params_before, params_after)
)
print(f"\nWeights changed: {weights_changed}")
assert weights_changed, "Parameters didn't change — gradient flow is broken!"
print(f"SGD optimizer step PASSED ✓")

Loss: 3.6259

Weight changes after one SGD step (lr=0.01):
  Layer 0: mean |delta| = 0.000287
  Layer 1: mean |delta| = 0.000388
  Layer 2: mean |delta| = 0.000395
  Layer 3: mean |delta| = 0.000531
  Layer 4: mean |delta| = 0.001329
  Layer 5: mean |delta| = 0.001611

Weights changed: True
SGD optimizer step PASSED ✓


---
## Step 7 — Test Parameter Serialization (FedAvg Round-Trip)

This tests the core mechanism that FedAvg relies on:
1. `get_model_params()` — client extracts weights → simulates uploading
2. `set_model_params()` — server loads weights into model → simulates broadcasting

We verify that extracting and re-loading produces identical outputs.

In [12]:
# ── Extract params from model A ───────────────────────────────────
model_A = create_model(device=DEVICE)
params_A = get_model_params(model_A)

print(f"Extracted {len(params_A)} parameter arrays:")
for i, p in enumerate(params_A):
    print(f"  [{i}] shape={p.shape}, dtype={p.dtype}")

# ── Load into a fresh model B ─────────────────────────────────────
model_B = create_model(device=DEVICE)

# Before loading — models should differ (different random init)
params_B_before = get_model_params(model_B)
differ_before = any(not np.allclose(a, b) for a, b in zip(params_A, params_B_before))
print(f"\nBefore set_model_params: models differ = {differ_before}  (expected: True)")

# Load A's weights into B
set_model_params(model_B, params_A)

# After loading — models should be identical
params_B_after = get_model_params(model_B)
match_after = all(np.allclose(a, b) for a, b in zip(params_A, params_B_after))
print(f"After set_model_params:  models match = {match_after}  (expected: True)")

# Verify identical outputs on same input
x_test = torch.randn(4, 17).to(DEVICE)
model_A.eval()
model_B.eval()
with torch.no_grad():
    out_A = model_A(x_test)
    out_B = model_B(x_test)
outputs_match = torch.allclose(out_A, out_B)
print(f"Identical outputs:       {outputs_match}  (expected: True)")

assert match_after, "Params were not loaded correctly!"
assert outputs_match, "Models produce different outputs after loading same params!"
print(f"\nParameter serialization round-trip PASSED ✓")

Extracted 6 parameter arrays:
  [0] shape=(128, 17), dtype=float32
  [1] shape=(128,), dtype=float32
  [2] shape=(64, 128), dtype=float32
  [3] shape=(64,), dtype=float32
  [4] shape=(8, 64), dtype=float32
  [5] shape=(8,), dtype=float32

Before set_model_params: models differ = True  (expected: True)
After set_model_params:  models match = True  (expected: True)
Identical outputs:       True  (expected: True)

Parameter serialization round-trip PASSED ✓


---
## Step 8 — Simulated FedAvg Aggregation (2 Clients)

This simulates the server-side aggregation step of FedAvg with 2 clients.
The server computes a weighted average of client parameters:

$$\theta_{global} = \sum_{k=1}^{K} \frac{n_k}{n_{total}} \cdot \theta_k$$

where $n_k$ is the number of samples on client $k$.

In [13]:
# ── Create two "client" models with different params ───────────────
client_a = create_model(device=DEVICE)
client_b = create_model(device=DEVICE)

params_a = get_model_params(client_a)
params_b = get_model_params(client_b)

# Simulate: client A has 4000 samples, client B has 2000
n_a, n_b = 4000, 2000
n_total = n_a + n_b

# ── Weighted average (this is the core of FedAvg) ─────────────────
avg_params = []
for pa, pb in zip(params_a, params_b):
    weighted_avg = (n_a / n_total) * pa + (n_b / n_total) * pb
    avg_params.append(weighted_avg)

# ── Load into a global model ──────────────────────────────────────
global_model = create_model(device=DEVICE)
set_model_params(global_model, avg_params)

# ── Verify the average is correct ─────────────────────────────────
expected = (n_a / n_total) * params_a[0] + (n_b / n_total) * params_b[0]
actual = get_model_params(global_model)[0]
assert np.allclose(expected, actual, atol=1e-6), "Aggregation mismatch!"

print(f"Client A samples: {n_a}, weight: {n_a/n_total:.2f}")
print(f"Client B samples: {n_b}, weight: {n_b/n_total:.2f}")
print(f"Aggregated {len(avg_params)} parameter tensors via weighted average")
print(f"\nFedAvg aggregation simulation PASSED ✓")

Client A samples: 4000, weight: 0.67
Client B samples: 2000, weight: 0.33
Aggregated 6 parameter tensors via weighted average

FedAvg aggregation simulation PASSED ✓


---
## Summary

### What was verified

| Test | Status |
|------|--------|
| Model instantiation and architecture inspection | ✓ |
| Per-layer parameter breakdown and communication cost | ✓ |
| Forward pass with random data | ✓ |
| Forward pass with real preprocessed data | ✓ |
| Weighted CrossEntropyLoss computation | ✓ |
| Backward pass + SGD optimizer step | ✓ |
| Parameter serialization round-trip (get → set) | ✓ |
| Simulated FedAvg weighted aggregation (2 clients) | ✓ |

### Communication cost (for thesis)
The model has ~11,000 parameters × 4 bytes = ~44 KB per update.
Per round with full participation:
- K=5: ~440 KB (5 clients × 44 KB × 2 directions)
- K=10: ~880 KB

### Output file
- `src/model.py` — ready for use by `federated.py` and `baselines.py`

### Next step
→ **FedAvg implementation** (`federated.py`): Build the server + client training loop using the partition files and this model.